In [1]:
import sys, os, glob, shutil

print("=" * 70)
print("STEP 1: PLATFORM DETECTION & PERMANENT CACHE")
print("=" * 70)

PERSISTENT_DIR = "/kaggle/working"
OUTPUT_BASE = "/kaggle/working"

# --- MOUNT PERMANENT MODEL CACHE ---
MOUNTED_CACHE = "/kaggle/input/datasets/bahaeeddineessadki/marker-model-cache"
LOCAL_CACHE = os.path.join(PERSISTENT_DIR, "marker_models_cache")

# Check if cache is already populated from a previous run this session
CACHE_MARKER = os.path.join(LOCAL_CACHE, ".cache_ready")
MODELS_CACHED = False

if os.path.exists(CACHE_MARKER):
    print(f"\n✅ Local cache already populated — skipping copy.")
    MODELS_CACHED = True
elif os.path.exists(MOUNTED_CACHE):
    mounted_items = os.listdir(MOUNTED_CACHE)
    if len(mounted_items) == 0:
        print(f"\n⚠️  Dataset mounted but empty! Models will download on first use.")
    else:
        print(f"\n📦 Found permanent dataset: {MOUNTED_CACHE}")

        # Copy (more reliable than symlink for HF cache)
        if os.path.islink(LOCAL_CACHE):
            os.unlink(LOCAL_CACHE)
        elif os.path.isdir(LOCAL_CACHE):
            shutil.rmtree(LOCAL_CACHE)

        print("   Copying to local cache folder... (takes ~30 sec for 3GB)")
        shutil.copytree(MOUNTED_CACHE, LOCAL_CACHE)
        print(f"✅ Copied to: {LOCAL_CACHE}")
        print("   Models ready — NO DOWNLOAD NEEDED!")

        # Write marker so subsequent runs skip the copy
        with open(CACHE_MARKER, "w") as f:
            f.write("ready")
        MODELS_CACHED = True
else:
    print(f"\n⚠️  Dataset NOT mounted!")
    print("   Click 'Add Data' → 'Your Datasets' → 'marker-model-cache'")
    print("   Then re-run this cell.")
    os.makedirs(LOCAL_CACHE, exist_ok=True)

# Set env vars
os.environ["HF_HOME"] = LOCAL_CACHE
os.environ["TRANSFORMERS_CACHE"] = os.path.join(LOCAL_CACHE, "transformers")
os.environ["TORCH_HOME"] = os.path.join(LOCAL_CACHE, "torch")
os.environ["XDG_CACHE_HOME"] = LOCAL_CACHE

# GPU Check
try:
    import torch
    print(f"\n🔥 PyTorch: {torch.__version__}")
    print(f"📟 CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("\n⚠️  PyTorch not installed yet.")

print("\n✅ Setup complete.")

STEP 1: PLATFORM DETECTION & PERMANENT CACHE

📦 Found permanent dataset: /kaggle/input/datasets/bahaeeddineessadki/marker-model-cache
   Copying to local cache folder... (takes ~30 sec for 3GB)
✅ Copied to: /kaggle/working/marker_models_cache
   Models ready — NO DOWNLOAD NEEDED!

🔥 PyTorch: 2.10.0+cu128
📟 CUDA: True
🎮 GPU: Tesla T4
💾 VRAM: 15.6 GB

✅ Setup complete.


In [2]:
import os

print("\n" + "=" * 70)
print("STEP 2: INSTALL MARKER")
print("=" * 70)

# Define CACHE_DIR if missing (e.g., after kernel restart)
try:
    CACHE_DIR
except NameError:
    CACHE_DIR = "/kaggle/working/marker_models_cache"
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f"⚠️  CACHE_DIR not found. Using default: {CACHE_DIR}")

os.environ["HF_HOME"] = CACHE_DIR
#os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers") #could be deleted based on claude

try:
    import marker
    print(f"\u2705 Marker already installed (v{marker.__version__})")
except ImportError:
    print("\u23f3 Installing marker-pdf...")
    !pip install marker-pdf -q
    print("\u2705 Marker installed.")

!which marker_single
print("\n\u2705 Ready.")


STEP 2: INSTALL MARKER
⚠️  CACHE_DIR not found. Using default: /kaggle/working/marker_models_cache
/usr/local/bin/marker_single

✅ Marker installed.


In [3]:
print("\n" + "=" * 70)
print("STEP 3: CACHE STATUS")
print("=" * 70)

# Re-check cache
cache_size_gb = 0.0
if os.path.exists(CACHE_DIR):
    cache_size_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, filenames in os.walk(CACHE_DIR)
        for f in filenames
    ) / (1024 ** 3)

if cache_size_gb > 1.0:
    print(f"\n✅ Found {cache_size_gb:.1f} GB cached models.")
    print("   marker_single will load from cache instantly.")
else:
    print(f"\n⚠️  Cache empty ({cache_size_gb:.1f} GB).")
    print("   Models will auto-download during extraction (Cell 5).")
    print("   This is normal on first run — takes ~3-5 minutes.")

print(f"\n📂 Cache location: {CACHE_DIR}")
print("✅ Ready for PDF processing.")


STEP 3: CACHE STATUS

✅ Found 3.3 GB cached models.
   marker_single will load from cache instantly.

📂 Cache location: /kaggle/working/marker_models_cache
✅ Ready for PDF processing.


# <span style="font-size: 50px;">🛑 🚨 ⚠️ **CRITICAL WARNING** ⚠️ 🚨 🛑</span>

<div style="font-size: 24px; color: #cc0000; border-left: 8px solid #cc0000; padding: 15px; background-color: #fff5f5; margin-top: 10px;">

**STOP BEFORE RUNNING:** 

This cell or notebook contains actions that require your absolute attention. 
* Do not proceed without checking THE SETTINGS BLOCK BELOW .
* Proceeding may overwrite existing data files.
</div>


In [4]:
# ==================== CONFIG ====================
MODE = "auto"                  # "default" | "split" | "folder" | "auto"
SPLIT_AT_PAGE = 12             # For "split" mode (0 = auto half)
BATCH_INPUT_DIR = "/kaggle/working/pdf_batch"

# Single-file link (used when MODE is "default" or "split")
DRIVE_LINK = ""

# Folder link (used when MODE is "folder")
DRIVE_FOLDER_LINK = "https://drive.google.com/drive/folders/1Ke8QpXvI2iVkSHunL6NtKMO6ShZPd9ce?usp=drive_link"

# Output naming
ZIP_NAME = "output"
ADD_PAGE_MARKERS = False
# =================================================

# --- Auto-detect MODE from link if set to "auto" ---
if MODE == "auto":
    if DRIVE_FOLDER_LINK and "/folders/" in DRIVE_FOLDER_LINK:
        MODE = "folder"
        print("  \u2192 Auto-detected MODE = folder (from DRIVE_FOLDER_LINK)")
    elif DRIVE_LINK and ("/d/" in DRIVE_LINK or "id=" in DRIVE_LINK):
        MODE = "default"
        print("  \u2192 Auto-detected MODE = default (from DRIVE_LINK)")
    else:
        MODE = "folder"
        print("  \u2192 No link set, defaulting to MODE = folder")

print()
print("=" * 60)
print("CURRENT CONFIG")
print("=" * 60)
print(f"  MODE              = {MODE}")
print(f"  SPLIT_AT_PAGE     = {SPLIT_AT_PAGE}")
print(f"  BATCH_INPUT_DIR   = {BATCH_INPUT_DIR}")
print(f"  DRIVE_LINK        = {'<set>' if DRIVE_LINK else '<empty>'}")
print(f"  DRIVE_FOLDER_LINK = {'<set>' if DRIVE_FOLDER_LINK else '<empty>'}")
print(f"  ZIP_NAME          = {ZIP_NAME}")
print(f"  ADD_PAGE_MARKERS  = {ADD_PAGE_MARKERS}")
print("=" * 60)
print()

# Strip spaces and force lowercase to handle typos or accidental spacing
user_input = input("Review the config above. Type 'continue' to proceed: ").strip().lower()

while user_input != "continue":
    print("Invalid input. You must type 'continue' exactly to unlock the notebook.")
    user_input = input("Review the config above. Type 'continue' to proceed: ").strip().lower()

print("Validation successful. Proceeding with execution...")

⚠️ CRITICAL: Review the config cell below. Type 'continue' to proceed:  continue


✅ Validation successful. Proceeding with execution...


In [5]:
# Config moved to Cell 4. This cell kept for backward compatibility.

In [6]:
import urllib.request, os, re

print("\n" + "=" * 70)
print("STEP 4: DOWNLOAD PDF(s)")
print("=" * 70)



os.makedirs(BATCH_INPUT_DIR, exist_ok=True)

# --- SINGLE FILE ---
if MODE == "default" or MODE== "split":
    match = re.search(r'/d/([a-zA-Z0-9_-]+)|id=([a-zA-Z0-9_-]+)', DRIVE_LINK)
    if not match:
        raise ValueError("Invalid Google Drive file link.")
    FILE_ID = match.group(1) if match.group(1) else match.group(2)
    PDF_URL = f"https://drive.google.com/uc?export=download&id={FILE_ID}"
    
    timestamp = __import__('datetime').datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_name = f"doc_{timestamp}.pdf"
    PDF_PATH = os.path.join(os.path.dirname(BATCH_INPUT_DIR), pdf_name)  # /kaggle/working/
    
    print(f"⬇️  Downloading single file...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    
    if os.path.exists(PDF_PATH) and os.path.getsize(PDF_PATH) > 1000:
        print(f"✅ Saved: {os.path.basename(PDF_PATH)} ({os.path.getsize(PDF_PATH)/1024/1024:.2f} MB)")
    else:
        raise FileNotFoundError("PDF download failed. Check Drive sharing.")

# ---  FOLDER ---
elif DRIVE_FOLDER_LINK.strip():
    try:
        import gdown
    except ImportError:
        !pip install gdown -q
        import gdown
    
    print(f"📁 Downloading folder...")
    folder_match = re.search(r'/folders/([a-zA-Z0-9_-]+)', DRIVE_FOLDER_LINK)
    if not folder_match:
        raise ValueError("Invalid Google Drive folder link.")
    
    folder_id = folder_match.group(1)
    !gdown --folder "https://drive.google.com/drive/folders/{folder_id}" -O "{BATCH_INPUT_DIR}" --remaining-ok
    
    # Flatten: move any nested PDFs to root of BATCH_INPUT_DIR
    for root, dirs, files in os.walk(BATCH_INPUT_DIR):
        for f in files:
            if f.lower().endswith('.pdf'):
                src = os.path.join(root, f)
                dst = os.path.join(BATCH_INPUT_DIR, f)
                if src != dst:
                    import shutil
                    shutil.move(src, dst)
    # Remove empty subfolders
    for root, dirs, files in os.walk(BATCH_INPUT_DIR, topdown=False):
        for d in dirs:
            dpath = os.path.join(root, d)
            if not os.listdir(dpath):
                os.rmdir(dpath)
    
    pdf_files = [f for f in os.listdir(BATCH_INPUT_DIR) if f.lower().endswith('.pdf')]
    print(f"✅ Downloaded {len(pdf_files)} PDF(s) to {BATCH_INPUT_DIR}")
    for f in sorted(pdf_files):
        print(f"   {f}")

else:
    raise ValueError("Set either DRIVE_LINK or DRIVE_FOLDER_LINK")


STEP 4: DOWNLOAD PDF(s)
📁 Downloading folder...
Retrieving folder contents
Processing file 1Fo72Kl0hgDAVbAKhmudTZK5aEFJTG7Jy DOC-20260326-WA0034.pdf
Processing file 1q4JZyxUYCSQvCh7466SyjcyTx3u7xzXW Jews and Muslims in the Maghreb (Version anglaise) .pdf
Processing file 1QHq0Tm0PXFVpZcsySEV4UbwMMXj_0IiO سيكولوجية الصحة كامل 2025-2026-1.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1Fo72Kl0hgDAVbAKhmudTZK5aEFJTG7Jy
To: /kaggle/working/pdf_batch/DOC-20260326-WA0034.pdf
100%|███████████████████████████████████████| 1.09M/1.09M [00:00<00:00, 110MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q4JZyxUYCSQvCh7466SyjcyTx3u7xzXW
To: /kaggle/working/pdf_batch/Jews and Muslims in the Maghreb (Version anglaise) .pdf
100%|█████████████████████████████████████████| 444k/444k [00:00<00:00, 116MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QHq0Tm0PXFVpZcsySEV4UbwMM

In [ ]:
#step 5
import time, os, glob, subprocess, math, shutil, datetime, shlex, re, json
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

print("\n" + "=" * 70)
print("STEP 5: RUNNING MARKER EXTRACTION")
print("=" * 70)


OUTPUT_DIR = os.path.join("/kaggle/working", "marker_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    PDF_PATH
except NameError:
    existing = glob.glob("/kaggle/working/*.pdf")
    PDF_PATH = existing[0] if existing else None

import torch
gpu_count = torch.cuda.device_count()
print(f"🎮 GPUs: {gpu_count}")
print(f"⚙️  Mode: {MODE}\n")

start = time.time()

# ---------------------------------------------------------------------
# PERSISTENT WORKER REGISTRY (Prevents model reloading across executions)
# ---------------------------------------------------------------------
if "_MARKER_WORKERS" not in globals():
    _MARKER_WORKERS = {}

def get_marker_worker(device):
    global _MARKER_WORKERS
    # Re-use worker if it's already active and resident in GPU VRAM
    if device in _MARKER_WORKERS:
        proc = _MARKER_WORKERS[device]
        if proc.poll() is None:
            return proc

    # Inline worker loop that keeps models warm in VRAM across calls
    worker_code = """import sys, os, json
import marker.models

# --- FIX: patch marker.models BEFORE importing convert_single ---
# convert_single (or something it imports) very likely does
# "from marker.models import create_model_dict" at its own top level.
# That statement copies a direct reference to the ORIGINAL function into
# convert_single's namespace at import time. If we patch marker.models
# AFTER that import already happened, convert_single's copy is left
# untouched and every job call reloads a brand-new model set on top of
# the one loaded during warmup -> VRAM grows every call -> CUDA OOM.
# Patching first means any "from marker.models import X" that happens
# during or after this point binds to OUR cached wrapper instead.

orig_create_model_dict = getattr(marker.models, 'create_model_dict', None)
orig_load_all_models = getattr(marker.models, 'load_all_models', None)

cached_models = None

def patched_create_model_dict(*args, **kwargs):
    global cached_models
    if cached_models is None:
        cached_models = orig_create_model_dict(*args, **kwargs)
    return cached_models

def patched_load_all_models(*args, **kwargs):
    global cached_models
    if cached_models is None:
        cached_models = orig_load_all_models(*args, **kwargs)
    return cached_models

if orig_create_model_dict:
    marker.models.create_model_dict = patched_create_model_dict
if orig_load_all_models:
    marker.models.load_all_models = patched_load_all_models

# NOW it's safe to import convert_single: it will see the patched functions.
try:
    from marker.scripts.convert_single import convert_single_cli as run_cli
except ImportError:
    from convert_single import main as run_cli

# Belt-and-suspenders: if convert_single (or a module it already imported
# earlier in the process) grabbed its own reference some other way, patch
# that module's namespace directly too, so it can't bypass the cache.
import sys as _sys
for _modname in ("marker.scripts.convert_single", "convert_single"):
    _mod = _sys.modules.get(_modname)
    if _mod is not None:
        if hasattr(_mod, "create_model_dict"):
            _mod.create_model_dict = patched_create_model_dict
        if hasattr(_mod, "load_all_models"):
            _mod.load_all_models = patched_load_all_models

print("🤖 Warm up: Caching marker models to GPU VRAM...", flush=True)
if orig_create_model_dict:
    try: patched_create_model_dict()
    except Exception: pass
if orig_load_all_models:
    try: patched_load_all_models()
    except Exception: pass

print("__WORKER_READY__", flush=True)

while True:
    line = sys.stdin.readline()
    if not line:
        break
    line = line.strip()
    if not line:
        continue
    try:
        data = json.loads(line)
        sys.argv = ["marker_single", data["pdf_path"], "--output_dir", data["out_dir"]]
        try:
            run_cli()
        except SystemExit as e:
            if e.code != 0:
                raise RuntimeError(f"CLI exited with code {e.code}")
        print("__WORKER_SUCCESS__", flush=True)
    except Exception as e:
        print(f"__WORKER_ERROR__: {str(e)}", flush=True)
"""
    worker_file = f"/kaggle/working/persistent_marker_worker_{device or 'default'}.py"
    with open(worker_file, "w", encoding="utf-8") as f:
        f.write(worker_code)

    env = os.environ.copy()
    if device is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(device)

    print(f"🚀 Initializing persistent worker for GPU {device or 'default'} (Models will load ONCE)...")
    proc = subprocess.Popen(
        [sys.executable, "-u", worker_file],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    # Keep track of warmup logs
    init_logs = []
    while True:
        line = proc.stdout.readline()
        if not line:
            print("".join(init_logs))
            raise RuntimeError(f"Worker for device {device} exited unexpectedly during warmup.")
        init_logs.append(line)
        cleaned = line.strip()
        if cleaned and not cleaned.startswith("__"):
            print(f"  [Worker GPU {device or 'default'}] {cleaned}")
        if "__WORKER_READY__" in line:
            break

    _MARKER_WORKERS[device] = proc
    return proc

# ---------------------------------------------------------------------
# PROGRESS-AWARE marker RUNNER (Leverages persistent VRAM workers)
# ---------------------------------------------------------------------
_PROGRESS_RE = re.compile(r'^(?P<desc>[^:]+):\s+(?P<pct>\d+)%\|.*?\|\s*(?P<n>\d+)/(?P<total>\d+)')
_LOADED_RE = re.compile(r'^Loaded (.+?) model')

def run_marker(pdf_path, out_dir, device=None, label=None, verbose_milestones=True):
    """Runs marker inside a long-lived process, showing a live tqdm bar instead of
    dumping raw output. On failure, prints the FULL captured log + raises."""
    name = label or os.path.basename(pdf_path)
    
    # Direct the file processing target to our active VRAM worker
    proc = get_marker_worker(device)
    
    # Pass execution targets securely to the worker via JSON stream
    job_data = json.dumps({"pdf_path": os.path.abspath(pdf_path), "out_dir": os.path.abspath(out_dir)})
    proc.stdin.write(job_data + "\n")
    proc.stdin.flush()

    log_lines = []
    seen_loaded = set()
    bar = None
    current_stage = None
    t0 = time.time()
    success = False

    try:
        while True:
            line = proc.stdout.readline()
            if not line:
                break
            log_lines.append(line)
            line = line.strip()
            if not line:
                continue

            if "__WORKER_SUCCESS__" in line:
                success = True
                break
            if "__WORKER_ERROR__" in line:
                print(f"\n❌ [{name}] worker processing FAILED — full log:\n")
                print("".join(log_lines))
                raise RuntimeError(f"{name}: marker processing failed: {line}")

            m = _LOADED_RE.match(line)
            if m and verbose_milestones and m.group(1) not in seen_loaded:
                seen_loaded.add(m.group(1))
                tqdm.write(f"  ✓ [{name}] loaded {m.group(1)} model")
                continue

            m = _PROGRESS_RE.match(line)
            if m:
                stage, n, total = m.group('desc'), int(m.group('n')), int(m.group('total'))
                if stage != current_stage:
                    if bar is not None:
                        bar.close()
                    bar = tqdm(total=total, desc=f"[{name}] {stage}", leave=False)
                    current_stage = stage
                bar.n = n
                bar.refresh()
    finally:
        if bar is not None:
            bar.close()

    elapsed = time.time() - t0

    if not success:
        print(f"\n❌ [{name}] marker worker crashed or disconnected — full log:\n")
        print("".join(log_lines))
        raise RuntimeError(f"{name}: marker worker disconnected unexpectedly.")

    print(f"  ✅ [{name}] done in {elapsed:.1f}s")

# -------------------------------------------------------------------------
# MODE 1 & 2: DEFAULT / BATCH_MULTIPLIER
# -------------------------------------------------------------------------
if MODE in ("default", "batch_multiplier"):
    if not PDF_PATH:
        raise NameError("No PDF found. Run Cell 4 first.")
    print(f"📄 {os.path.basename(PDF_PATH)}")
    run_marker(PDF_PATH, OUTPUT_DIR, label=os.path.basename(PDF_PATH))

    pdf_base = os.path.splitext(os.path.basename(PDF_PATH))[0]
    output_path = os.path.join(OUTPUT_DIR, pdf_base)

# -------------------------------------------------------------------------
# MODE 3: SPLIT (2 GPUs parallel)
# -------------------------------------------------------------------------
elif MODE == "split":
    if not PDF_PATH:
        raise NameError("No PDF found. Run Cell 4 first.")

    if gpu_count < 2:
        print("⚠️  Only 1 GPU. Falling back to default.")
        run_marker(PDF_PATH, OUTPUT_DIR, label=os.path.basename(PDF_PATH))
        pdf_base = os.path.splitext(os.path.basename(PDF_PATH))[0]
        output_path = os.path.join(OUTPUT_DIR, pdf_base)
    else:
        try:
            from pypdf import PdfReader, PdfWriter
        except ImportError:
            !pip install pypdf -q
            from pypdf import PdfReader, PdfWriter

        reader = PdfReader(PDF_PATH)
        total_pages = len(reader.pages)
        split_page = SPLIT_AT_PAGE if (0 < SPLIT_AT_PAGE < total_pages) else math.ceil(total_pages / 2)

        print(f"✂️  Split at page {split_page} / {total_pages}")
        print(f"   Part 1: pages 1-{split_page}")
        print(f"   Part 2: pages {split_page+1}-{total_pages}")

        part1 = os.path.join("/kaggle/working", "split_part1.pdf")
        part2 = os.path.join("/kaggle/working", "split_part2.pdf")

        w1, w2 = PdfWriter(), PdfWriter()
        for i, page in enumerate(reader.pages):
            (w1 if i < split_page else w2).add_page(page)
        with open(part1, "wb") as f: w1.write(f)
        with open(part2, "wb") as f: w2.write(f)

        out1 = os.path.join(OUTPUT_DIR, "_raw_part1")
        out2 = os.path.join(OUTPUT_DIR, "_raw_part2")

        print("\n🚀 GPU 0 + GPU 1 in parallel...\n")
        with ThreadPoolExecutor(max_workers=2) as ex:
            f1 = ex.submit(run_marker, part1, out1, "0", "GPU0")
            f2 = ex.submit(run_marker, part2, out2, "1", "GPU1")
            f1.result(); f2.result()

        md1 = glob.glob(f"{out1}/**/*.md", recursive=True)
        md2 = glob.glob(f"{out2}/**/*.md", recursive=True)
        if not md1 or not md2:
            raise FileNotFoundError("Marker produced no markdown output. Check GPU errors above.")

        base_name = os.path.splitext(os.path.basename(PDF_PATH))[0]
        merged_dir = os.path.join(OUTPUT_DIR, base_name)
        os.makedirs(merged_dir, exist_ok=True)

        merged_md = os.path.join(merged_dir, f"{base_name}.md")
        with open(merged_md, "w", encoding="utf-8") as out_f:
            out_f.write(f"{'='*50}\n📄 PART 1 — PAGES 1-{split_page}\n{'='*50}\n\n")
            with open(md1[0], "r", encoding="utf-8") as f: out_f.write(f.read())
            out_f.write(f"\n\n{'='*50}\n📄 PART 2 — PAGES {split_page+1}-{total_pages}\n{'='*50}\n\n")
            with open(md2[0], "r", encoding="utf-8") as f: out_f.write(f.read())

        for src_dir in [out1, out2]:
            for f in glob.glob(f"{src_dir}/**/*", recursive=True):
                if os.path.isfile(f) and not f.endswith('.md'):
                    shutil.copy2(f, merged_dir)

        shutil.rmtree(out1, ignore_errors=True)
        shutil.rmtree(out2, ignore_errors=True)
        os.remove(part1); os.remove(part2)

        output_path = merged_dir
        print(f"\n📄 Merged: {output_path}")

# -------------------------------------------------------------------------
# MODE 4: FOLDER (distribute across 2 GPUs)
# -------------------------------------------------------------------------
elif MODE == "folder":
    pdf_files = sorted([os.path.join(BATCH_INPUT_DIR, f) for f in os.listdir(BATCH_INPUT_DIR) if f.lower().endswith('.pdf')])
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs in {BATCH_INPUT_DIR}")

    print(f"📁 {len(pdf_files)} PDF(s)")

    container_name = f"batch_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    container_dir = os.path.join(OUTPUT_DIR, container_name)
    os.makedirs(container_dir, exist_ok=True)
    print(f"📦 Container: {container_name}/")

    def process_files(files, device, label):
        for pdf in files:
            name = os.path.splitext(os.path.basename(pdf))[0]
            out = os.path.join(container_dir, name)
            run_marker(pdf, out, device=device, label=f"{label}-{name}")

    if gpu_count >= 2 and len(pdf_files) >= 2:
        mid = (len(pdf_files) + 1) // 2
        g1, g2 = pdf_files[:mid], pdf_files[mid:]

        print("\n🚀 GPU 0 + GPU 1 in parallel...\n")
        with ThreadPoolExecutor(max_workers=2) as ex:
            f1 = ex.submit(process_files, g1, "0", "GPU0")
            f2 = ex.submit(process_files, g2, "1", "GPU1")
            f1.result(); f2.result()

        print(f"\n✅ Both GPUs finished")
    else:
        process_files(pdf_files, None, "CPU/GPU0")

    output_path = container_dir

else:
    raise ValueError(f"Unknown MODE: {MODE}")

elapsed = time.time() - start
print(f"\n{'='*70}")
print(f"✅ DONE in {elapsed:.1f} seconds ({elapsed/60:.1f} min)")
print(f"{'='*70}")


STEP 5: RUNNING MARKER EXTRACTION
🎮 GPUs: 2
⚙️  Mode: folder

📁 3 PDF(s)
📦 Container: batch_20260720_170530/

🚀 GPU 0 + GPU 1 in parallel...

🚀 Initializing persistent worker for GPU 0 (Models will load ONCE)...
🚀 Initializing persistent worker for GPU 1 (Models will load ONCE)...
  [Worker GPU 1] /usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  [Worker GPU 1] warnings.warn(
  [Worker GPU 0] /usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  [Worker GPU 0] warnings.warn(
  [Worker GPU 1] 🤖 Warm up: Caching marker models to GPU VRAM...
  [Worker GPU 0] 🤖 Warm up: Caching marker models to GPU VRAM...


[GPU0-DOC-20260326-WA0034] Recognizing Layout:   0%|          | 0/14 [00:00<?, ?it/s]

[GPU1-سيكولوجية الصحة كامل 2025-2026-1] Recognizing Layout:   0%|          | 0/22 [00:00<?, ?it/s]

[GPU0-DOC-20260326-WA0034] Running OCR Error Detection:   0%|          | 0/1 [00:00<?, ?it/s]

[GPU0-DOC-20260326-WA0034] Detecting bboxes:   0%|          | 0/1 [00:00<?, ?it/s]

...

[GPU0-DOC-20260326-WA0034] Recognizing Text:   0%|          | 0/142 [00:00<?, ?it/s]

  ✅ [GPU0-DOC-20260326-WA0034] done in 60.7s


[GPU0-Jews and Muslims in the Maghreb (Version anglaise) ] Recognizing Layout:   0%|          | 0/15 [00:00<?,…

[GPU0-Jews and Muslims in the Maghreb (Version anglaise) ] Running OCR Error Detection:   0%|          | 0/2 […

  ✅ [GPU0-Jews and Muslims in the Maghreb (Version anglaise) ] done in 12.3s


In [ ]:
import glob, os, shutil, re, datetime

print("\n" + "=" * 70)
print("STEP 6: FINALIZE OUTPUT")
print("=" * 70)



OUTPUT_DIR = os.path.join("/kaggle/working", "marker_output")
WORKING_DIR = "/kaggle/working"

# Recover output_path from Cell 5, or auto-detect
try:
    output_path
except NameError:
    subfolders = [f.path for f in os.scandir(OUTPUT_DIR) if f.is_dir()]
    if not subfolders:
        print("❌ No output found.")
        raise SystemExit
    output_path = max(subfolders, key=os.path.getctime)
    print(f"⚠️  Auto-detected: {output_path}")

md_files = glob.glob(f"{output_path}/*.md")
if not md_files:
    md_files = glob.glob(f"{output_path}/**/*.md", recursive=True)

# 1. Optional page markers
if ADD_PAGE_MARKERS and md_files:
    for md_path in md_files:
        with open(md_path, "r", encoding="utf-8") as f:
            content = f.read()
        page_pattern = r'!\[(.*?)\]\((_page_(\d+)_[^)]+)\)'
        seen = set()
        new_content = content
        for m in reversed(list(re.finditer(page_pattern, content))):
            p = int(m.group(3))
            if p not in seen:
                seen.add(p)
                marker = f"\n\n{'='*50}\n📄 PAGE {p + 1}\n{'='*50}\n\n"
                new_content = new_content[:m.start()] + marker + new_content[m.start():]
        if 0 not in seen and new_content.strip():
            new_content = f"{'='*50}\n📄 PAGE 1\n{'='*50}\n\n" + new_content
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(new_content)
    print(f"✅ Page markers added to {len(md_files)} file(s)")

# 2. Determine zip name
if ZIP_NAME.strip():
    zip_name = ZIP_NAME.strip()
    if not zip_name.endswith(".zip"):
        zip_name += ".zip"
else:
    zip_name = os.path.basename(output_path) + ".zip"

zip_path = os.path.join(WORKING_DIR, zip_name)

# 3. Zip
print(f"\n📦 Zipping: {os.path.basename(output_path)} → {zip_name}")
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', output_path)
print(f"✅ {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# 4. Clean loose folder
print(f"\n🧹 Cleaning up...")
shutil.rmtree(output_path)
print(f"   Deleted: {output_path}")

print(f"\n{'='*70}")
print(f"📥 FINAL OUTPUT: {zip_path}")
print(f"{'='*70}")

In [ ]:
##GITHUB_UPLOAD
import os, glob, subprocess, shutil
from IPython.display import display, HTML

print("\n" + "=" * 70)
print("STEP 7: PUSH TO GITHUB")
print("=" * 70)

# ==================== CONFIG ====================
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")  # Your Personal Access Token
GITHUB_USER = "freegamefire03-boop"       # e.g., "bahaeeddine"
REPO_NAME = "kaggle_marker_output"           # e.g., "pdf-extracts"
BRANCH = "main"                            # or "master"
COMMIT_MSG = "default"
# =================================================

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
TEMP_DIR = "/tmp/github_push"
WORKING_DIR = "/kaggle/working"

# 1. Find ONLY zip files in /kaggle/working root (excludes all subfolders)
zip_files = sorted(glob.glob(os.path.join(WORKING_DIR, "*.zip")))

if not zip_files:
    print("❌ No zip files found in /kaggle/working/")
    print("   Run Cell 6 first to generate output zips.")
    raise SystemExit

print(f"📦 Found {len(zip_files)} zip(s) to push:")
for z in zip_files:
    size = os.path.getsize(z) / 1024 / 1024
    print(f"   {os.path.basename(z)} ({size:.1f} MB)")

# 2. Clone repo into temp folder
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

print(f"\n⬇️  Cloning {REPO_NAME}...")
result = subprocess.run(["git", "clone", REPO_URL, TEMP_DIR], capture_output=True, text=True)
if result.returncode != 0 and "empty repository" not in result.stderr.lower():
    print(f"⚠️  Clone issue: {result.stderr}")
    print("   Attempting fresh init...")
    os.makedirs(TEMP_DIR, exist_ok=True)
    subprocess.run(["git", "init"], cwd=TEMP_DIR, check=True)
    subprocess.run(["git", "remote", "add", "origin", REPO_URL], cwd=TEMP_DIR, check=True)
    subprocess.run(["git", "checkout", "-b", BRANCH], cwd=TEMP_DIR, check=True)
else:
    print(f"✅ Cloned to {TEMP_DIR}")

# 3. Copy ONLY zip files (no other folders leak in)
print(f"\n📤 Copying zips to repo...")
for z in zip_files:
    shutil.copy2(z, TEMP_DIR)
    print(f"   + {os.path.basename(z)}")

# 4. Git commit & push
print(f"\n🚀 Pushing to {BRANCH}...")
subprocess.run(["git", "config", "user.email", "kaggle@bot.com"], cwd=TEMP_DIR, check=True)
subprocess.run(["git", "config", "user.name", "Kaggle Bot"], cwd=TEMP_DIR, check=True)
subprocess.run(["git", "add", "*.zip"], cwd=TEMP_DIR, check=True)

# Track if the process finishes successfully to display links
show_links = False

# Check if there's anything to commit
status = subprocess.run(["git", "status", "--porcelain"], cwd=TEMP_DIR, capture_output=True, text=True)
if not status.stdout.strip():
    print("⚠️  Nothing new to commit (zips already exist in repo).")
    show_links = True
else:
    subprocess.run(["git", "commit", "-m", COMMIT_MSG], cwd=TEMP_DIR, check=True)
    push_result = subprocess.run(["git", "push", "origin", BRANCH], cwd=TEMP_DIR, capture_output=True, text=True)
    if push_result.returncode == 0:
        print(f"\n{'='*70}")
        print(f"✅ SUCCESSFULLY PUSHED to github.com/{GITHUB_USER}/{REPO_NAME}")
        print(f"{'='*70}")
        show_links = True
    else:
        print(f"❌ Push failed: {push_result.stderr}")

# --- GENERATE AND DISPLAY CLICKABLE HTML LINKS ---
if show_links:
    repo_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}"
    
    # Generate download layout for all detected zip files
    download_links_html = ""
    for z in zip_files:
        filename = os.path.basename(z)
        # Using raw.githubusercontent.com forces the browser to download the file directly
        direct_download_url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{filename}"
        
        download_links_html += f"""
        <p style="margin: 10px 0;">
            <strong>💾 Direct Download ({filename}):</strong> <br>
            <a href="{direct_download_url}" target="_blank" style="color: #fff; background-color: #24292e; padding: 6px 12px; border-radius: 4px; text-decoration: none; font-size: 13px; font-weight: bold; display: inline-block; margin-top: 4px; border: 1px solid #1b1f23;">
                📥 Click to Download File
            </a>
        </p>
        """
        
    html_output = f"""
    <div style="padding: 15px; border: 1px solid #d1d5da; border-radius: 6px; background-color: #212529; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; margin-top: 20px;">
        <h3 style="color: #28a745; margin-top: 0; display: flex; align-items: center;">🚀 File Links Generated Successfully!</h3>
        
        <p style="margin: 10px 0;">
            <strong>📊 GitHub Repository:</strong> <br>
            <a href="{repo_url}" target="_blank" style="color: #0366d6; text-decoration: none; font-weight: bold; font-size: 14px;">
                🔗 View Repository UI
            </a>
        </p>
        
        <hr style="border: 0; border-top: 1px solid #e1e4e8; margin: 15px 0;">
        {download_links_html}
    </div>
    """
    display(HTML(html_output))

In [ ]:
# import os
# import json
# import subprocess

# print("=" * 70)
# print("SAVING MODELS TO PERMANENT KAGGLE DATASET")
# print("=" * 70)

# # Your Kaggle username (change this!)
# YOUR_USERNAME = "bahaeeddineessadki"  # <-- EDIT THIS

# CACHE_DIR = "/kaggle/working/marker_models_cache"
# DATASET_SLUG = "marker-model-cache"
# DATASET_ID = f"{YOUR_USERNAME}/{DATASET_SLUG}"

# # 1. Create metadata file
# meta = {
#     "title": "Marker PDF Model Cache",
#     "id": DATASET_ID,
#     "licenses": [{"name": "CC0-1.0"}]
# }

# meta_path = os.path.join(CACHE_DIR, "dataset-metadata.json")
# with open(meta_path, "w") as f:
#     json.dump(meta, f, indent=2)

# print(f"✅ Metadata created: {meta_path}")

# # 2. Create/update the dataset
# print(f"\n⬆️  Uploading ~3GB to Kaggle Dataset: {DATASET_ID}")
# print("   This takes 3-5 minutes. Do not interrupt.\n")

# result = subprocess.run(
#     ["kaggle", "datasets", "create", "-p", CACHE_DIR, "--dir-mode", "zip", "--public"],
#     capture_output=True, text=True
# )

# print(result.stdout)
# if result.stderr:
#     print("Stderr:", result.stderr)

# print(f"\n{'='*70}")
# print(f"✅ DATASET CREATED: {DATASET_ID}")
# print(f"{'='*70}")
# print("\n🔗 URL: https://www.kaggle.com/datasets/" + DATASET_ID)

In [ ]:
import os, signal, subprocess, time, gc

print("🧹 Freeing GPU VRAM from orphaned marker workers...")

# 1. Kill any workers tracked in this kernel's registry
killed = 0
if "_MARKER_WORKERS" in globals():
    for device, proc in list(_MARKER_WORKERS.items()):
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            killed += 1
            print(f"   killed tracked worker for device {device} (pid {proc.pid})")
    _MARKER_WORKERS.clear()

# 2. Catch any worker processes not tracked in this kernel session
#    (e.g. left over from a kernel restart / previous execution)
result = subprocess.run(["pgrep", "-f", "persistent_marker_worker"], capture_output=True, text=True)
stray_pids = [int(p) for p in result.stdout.split()]
for pid in stray_pids:
    try:
        os.kill(pid, signal.SIGKILL)
        killed += 1
        print(f"   killed stray worker pid {pid}")
    except ProcessLookupError:
        pass

if killed:
    time.sleep(2)  # give the driver a moment to reclaim memory after process exit

# 3. Clear any CUDA memory held directly by this notebook process, if any
try:
    import torch
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("   cleared this process's own CUDA cache")
except Exception as e:
    print(f"   (torch cleanup skipped: {e})")

print(f"\n✅ Done. Killed {killed} worker process(es).\n")

# 4. Show current VRAM state to confirm
subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"])